<a href="https://colab.research.google.com/github/Blackthornedejavre/GoogleColap/blob/main/Catastro/2_ProcesamientoDatosRecibidos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Agrupar por APN o Titular

Una vez recibido del servicio de montes la consulta catastral se debe hacer una copia en el siguiente  f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/2.Resultados Consulta Catastral Masiva/RC_{monte}.xlsx"

In [3]:
#Instalar Librerias
!pip install pandas
!pip install openpyxl

In [4]:
#Importar Librerias
import pandas as pd

In [5]:
# Variables
monte = "Santa_Eulalia_de_Carranzo"

In [6]:
# Rutas de archivos
directorio = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/2.Resultados Consulta Catastral Masiva/RC_{monte}.xlsx"

In [ ]:
# Leer el archivo Excel y obtener los nombres de las columnas
datos = pd.read_excel(directorio)
nombres_encabezados = datos.columns.tolist()

# Filtrar los nombres que comienzan por "NIF"
var_NIF = [encabezado for encabezado in nombres_encabezados if encabezado.startswith("NIF")]

# Solo tomar la segunda columna de la lista var_NIF
if len(var_NIF) > 1:
    var_NIF = [var_NIF[1]]
print(var_NIF)

# Filtrar los nombres que corresponden a las columnas a mostrar
columnas_mostrar = ['RC', 'SUF', 'APN', 'DFT1', 'DFT2'] + var_NIF

# Filtrar las filas donde la columna 'APN' no es nula
datos_filtrados = datos[columnas_mostrar].dropna(subset=['APN'])

# Agrupar por 'APN' y obtener una lista de 'RC' asociados a cada 'APN'
Info_Titulares = datos_filtrados.groupby('APN').agg({
    'RC': lambda x: ', '.join(sorted(set(x))),  # Combine unique 'RC' values into a comma-separated string
    'SUF': 'first',   # Take the first 'SUF' value for each 'APN'
    'DFT1': 'first',  # Take the first 'DFT1' value for each 'APN'
    'DFT2': 'first',  # Take the first 'DFT2' value for each 'APN'
    **{col: 'first' for col in var_NIF},  # Take the first value for each NIF column
}).reset_index()

# Mostrar el resultado
display(Info_Titulares)

# Group by y summarize en group_by_RF_2
group_by_RF_2 = datos[['RC', 'APN']].dropna(subset=['APN']).groupby('RC').agg({'APN': lambda x: ', '.join(pd.unique(x))}).reset_index()
UnaRCVariostitulares = group_by_RF_2.rename(columns={'APN': 'APNs'})
display(UnaRCVariostitulares)

# Guardado en archivos Excel
Guardar_excel_Info_Titulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_Info_Titulares_colab.xlsx"
Info_Titulares.to_excel(Guardar_excel_Info_Titulares, index=True)

Guardar_excel_UnaRCVariostitulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.xlsx"
UnaRCVariostitulares.to_excel(Guardar_excel_UnaRCVariostitulares, index= False)

# Guardado en archivos CSV
Guardar_csv_Info_Titulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_Info_Titulares_colab.csv"
Info_Titulares.to_csv(Guardar_csv_Info_Titulares, index=False)

Guardar_csv_UnaRCVariostitulares = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.csv"
UnaRCVariostitulares.to_csv(Guardar_csv_UnaRCVariostitulares, index=False)

## Categorizar Tipo de propiedad

Ahora se puede categorizar el tipo de propiedad en "Privada", " Publica" "Desconocida".Utiliza UnaRC asociada a varios titulares para obtener el tipo de propiedad.


In [10]:
import pandas as pd
import os

# Directory for Excel and CSV files
excel_file_path  = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/4.Filtrado datos/Filtrado_{monte}_UnaRCVariostitulares_colap.xlsx"

# Words dictionary
words_dict = {"VECINOS", "VECINAL", "CONFEDERACION", "CAMINOS", "AYUNTAMIENTO","JUNTA","DESCUENTO","PRINCIPADO DE ASTURIAS","COMUNAL","FERROVIARIAS","DEMARCACION DE CARRETERAS","DEMARCACION DE COSTAS"}

# Read the Excel file
Excel_UnifRC_colap = pd.read_excel(excel_file_path)

# Function to map APNs to Tipo_Propiedad
def map_tipo_propiedad(apn_value):
    if apn_value == "EN INVESTIGACION":
        return "DESCONOCIDO"
    elif any(word.lower() in apn_value.lower() for word in words_dict):
        return "PÚBLICA"
    else:
        return "PRIVADA"

# Create the new "Tipo_Propiedad" column based on the mapping function
Excel_UnifRC_colap["Tipo_Propiedad"] = Excel_UnifRC_colap["APNs"].map(map_tipo_propiedad)


# Directory for Excel and CSV files
output_directory = f"/content/drive/MyDrive/Catastro/Catastro_Analisis/Actuacion_Catastro/{monte}/5.Union_GIS"
output_directory_Excel = os.path.join(output_directory, f"Filtrado_{monte}_UnaRCVariostitulares_TP_colab.xlsx")
output_directory_CSV = os.path.join(output_directory, f"Filtrado_{monte}_UnaRCVariostitulares_TP_colab.csv")

# Save the DataFrame to Excel and CSV files
Excel_UnifRC_colap.to_excel(output_directory_Excel, index= False)
Excel_UnifRC_colap.to_csv(output_directory_CSV, index=False,encoding='utf-8 sig')